In [4]:
# imports and paths

from pathlib import Path

import sys
import numpy as np
import pandas as pd


# Locate the project root.
if Path.cwd().name in {"notebooks", "notebooks-phase2"}:
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "task7_xai"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Locate the supplied eLCS implementation.
ELCS_ROOT = (
    PROJECT_ROOT
    / "external"
    / "scikit-eLCS-master"
)

if str(ELCS_ROOT) not in sys.path:
    sys.path.insert(0, str(ELCS_ROOT))

from skeLCS import eLCS

# load the final training data

In [2]:
X_train = pd.read_csv(
    DATA_DIR
    / "X_train_preprocessed_undersampled.csv"
)

y_train = pd.read_csv(
    DATA_DIR
    / "y_train_preprocessed_undersampled.csv"
).squeeze()


X_train_array = X_train.to_numpy(
    dtype=float
)

y_train_array = y_train.to_numpy(
    dtype=int
)


print("Training shape:", X_train.shape)

print("\nClass distribution:")
print(
    y_train.value_counts().sort_index()
)

Training shape: (29713, 34)

Class distribution:
heart_disease_target
0    25000
1     4713
Name: count, dtype: int64


# Train one final improved eLCS

In [5]:
FINAL_SEED = 3

final_model = eLCS(
    N=8000,
    learning_iterations=50000,
    random_state=FINAL_SEED,
)

final_model.fit(
    X_train_array,
    y_train_array,
)

print("Final improved eLCS trained.")
print("Seed:", FINAL_SEED)

Final improved eLCS trained.
Seed: 3


# debugging


In [11]:
population = final_model.population

rules = population.popSet


positive_rule_indices = [
    index
    for index, rule in enumerate(rules)
    if int(rule.phenotype) == 1
]

negative_rule_indices = [
    index
    for index, rule in enumerate(rules)
    if int(rule.phenotype) == 0
]


print(
    "Unique rules:",
    len(rules),
)

print(
    "Total classifiers including numerosity:",
    population.microPopSize,
)

print(
    "Class 0 rules:",
    len(negative_rule_indices),
)

print(
    "Class 1 rules:",
    len(positive_rule_indices),
)


Unique rules: 6997
Total classifiers including numerosity: 8000
Class 0 rules: 4428
Class 1 rules: 2569


## only class 1 / heart-disease rules
## sorted by Match Count, so the most experienced rules are at the top
## each rule's conditions, accuracy, fitness, and experience

In [16]:
# Get all class-1 rules from the trained population.
rule_rows = []

for index, rule in enumerate(final_model.population.popSet):

    if int(rule.phenotype) != 1:
        continue

    attribute_names = [
        X_train.columns[i]
        for i in rule.specifiedAttList
    ]

    rule_rows.append({
        "Rule": index,
        "Attributes": ", ".join(attribute_names),
        "Conditions": str(rule.condition),
        "Prediction": int(rule.phenotype),
        "Accuracy": rule.accuracy,
        "Fitness": rule.fitness,
        "Match Count": rule.matchCount,
        "Correct Count": rule.correctCount,
        "Numerosity": rule.numerosity,
    })


positive_rules_df = pd.DataFrame(rule_rows)

positive_rules_df = positive_rules_df.sort_values(
    by="Match Count",
    ascending=False,
)

display(
    positive_rules_df.head(10)
)

,Rule,Attributes,Conditions,Prediction,Accuracy,Fitness,Match Count,Correct Count,Numerosity
248,650,"alcohol_intake_binned, smoking_Current, high_b...","[1.0, 0.0, 1.0, 3.0, 1.0, 2.0, 1.0, 2.0, 1.0, ...",1,0.622708,0.093632,1418,883,1
271,699,"systolic_bp_binned, age_binned, gender, alcoho...","[2.0, 3.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0]",1,0.627702,0.097447,1249,784,1
242,637,"systolic_bp_binned, gender, alcohol_intake_bin...","[2.0, 1.0, 1.0, 2.0, 3.0, 1.0]",1,0.627641,0.097399,1136,713,1
363,888,"smoking_Current, age_binned, ldl_binned, diast...","[0.0, 3.0, 1.0, 1.0, 0.0, 1.0, 1.0, 2.0]",1,0.634015,0.102447,929,589,1
634,1497,"systolic_bp_binned, crp_level_binned, choleste...","[2.0, 2.0, 1.0, 0.0, 2.0, 1.0, 3.0, 0.0, 1.0]",1,0.639522,0.106974,921,589,1
301,765,"employment_status_Unemployed, gender, heart_ra...","[0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 2.0, 0.0, ...",1,0.675894,0.141056,867,586,1
632,1494,"age_binned, systolic_bp_binned, employment_sta...","[3.0, 2.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, ...",1,0.627240,0.097089,837,525,1
176,507,"smoking_Former, diabetes, diastolic_bp_binned,...","[0.0, 1.0, 1.0, 3.0, 2.0, 2.0, 0.0]",1,0.645161,0.111774,775,500,1
330,821,"hdl_binned, crp_level_binned, gender, diabetes...","[2.0, 2.0, 1.0, 1.0, 0.0, 1.0, 2.0, 0.0, 1.0]",1,0.631158,0.100159,751,474,1
315,794,"systolic_bp_binned, diastolic_bp_binned, age_b...","[2.0, 1.0, 3.0, 1.0, 2.0, 1.0, 0.0, 0.0]",1,0.658177,0.123513,746,491,1


In [17]:
for rule_number in [650, 699, 637]:
    rule = positive_rules_df[
        positive_rules_df["Rule"] == rule_number
    ].iloc[0]

    print(f"\nRule {rule_number}")
    print("Attributes:", rule["Attributes"])
    print("Conditions:", rule["Conditions"])
    print("Accuracy:", rule["Accuracy"])
    print("Fitness:", rule["Fitness"])
    print("Match Count:", rule["Match Count"])
    print("Correct Count:", rule["Correct Count"])


Rule 650
Attributes: alcohol_intake_binned, smoking_Current, high_blood_pressure, age_binned, ldl_binned, crp_level_binned, salt_intake_binned, systolic_bp_binned, diastolic_bp_binned, homocysteine_level_binned
Conditions: [1.0, 0.0, 1.0, 3.0, 1.0, 2.0, 1.0, 2.0, 1.0, 0.0]
Accuracy: 0.6227080394922426
Fitness: 0.09363158250931775
Match Count: 1418
Correct Count: 883

Rule 699
Attributes: systolic_bp_binned, age_binned, gender, alcohol_intake_binned, high_ldl_cholesterol, diastolic_bp_binned, salt_intake_binned, heart_rate_binned
Conditions: [2.0, 3.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0]
Accuracy: 0.6277021617293835
Fitness: 0.09744692103332393
Match Count: 1249
Correct Count: 784

Rule 637
Attributes: systolic_bp_binned, gender, alcohol_intake_binned, bmi_level, age_binned, salt_intake_binned
Conditions: [2.0, 1.0, 1.0, 2.0, 3.0, 1.0]
Accuracy: 0.6276408450704225
Fitness: 0.09739933514983444
Match Count: 1136
Correct Count: 713
